# StreamView Analytics
## 01 - Preprocesamiento de datos

### Descripción

En este notebook se realiza el proceso de carga, exploración inicial,
limpieza y preparación de los datos correspondientes al catálogo
audiovisual de StreamView Analytics.

El procesamiento tiene como propósito generar un conjunto de datos
consistente que permita posteriormente analizar la relación entre
género, popularidad e ingresos de las películas.

# Importacion de librerías y carga de datos

In [2]:
import pandas as pd
import numpy as np

df_movies = pd.read_csv("netflix_movies_detailed_up_to_2025.csv")

df_movies.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,language,description,popularity,vote_count,vote_average,budget,revenue
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,6.380,NaN,"Comedy, Adventure, Fantasy, Animation, Family",en,A bored and domesticated Shrek pacts with deal...,203.893,7449,6.380,165000000,752600867
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,8.369,NaN,"Action, Science Fiction, Adventure",en,"Cobb, a skilled thief who commits corporate es...",156.242,37119,8.369,160000000,839030630
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17,2010,7.744,NaN,"Adventure, Fantasy",en,"Harry, Ron and Hermione walk away from their l...",121.191,19327,7.744,250000000,954305868
3,38757,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24,2010,7.600,NaN,"Animation, Family, Adventure",en,"Feisty teenager Rapunzel, who has long and mag...",111.762,11638,7.600,260000000,592461732
4,10191,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18,2010,7.800,NaN,"Fantasy, Adventure, Animation, Family",en,As the son of a Viking leader on the cusp of m...,110.044,13259,7.800,165000000,494879471


# Conocer las dimensiones

In [3]:
print("Cantidad de filas:", df_movies.shape[0])
print("Cantidad de columnas:", df_movies.shape[1])

Cantidad de filas: 16000
Cantidad de columnas: 18


# Revisar las columnas

In [4]:
df_movies.columns

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'genres', 'language',
       'description', 'popularity', 'vote_count', 'vote_average', 'budget',
       'revenue'],
      dtype='object')

# Información general

In [5]:
df_movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 18 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   show_id       16000 non-null  int64  
 1   type          16000 non-null  object 
 2   title         16000 non-null  object 
 3   director      15868 non-null  object 
 4   cast          15796 non-null  object 
 5   country       15534 non-null  object 
 6   date_added    16000 non-null  object 
 7   release_year  16000 non-null  int64  
 8   rating        16000 non-null  float64
 9   duration      0 non-null      float64
 10  genres        15893 non-null  object 
 11  language      16000 non-null  object 
 12  description   15868 non-null  object 
 13  popularity    16000 non-null  float64
 14  vote_count    16000 non-null  int64  
 15  vote_average  16000 non-null  float64
 16  budget        16000 non-null  int64  
 17  revenue       16000 non-null  int64  
dtypes: float64(4), int64(5), o

# Estadística descriptiva

In [6]:
df_movies[
    ["release_year", "popularity", "vote_count",
     "vote_average", "budget", "revenue"]
].describe()

,release_year,popularity,vote_count,vote_average,budget,revenue
count,16000.000000,16000.000000,16000.000000,16000.000000,1.600000e+04,1.600000e+04
mean,2017.500000,20.384728,718.656125,5.956368,8.766792e+06,2.446308e+07
std,4.609916,68.610033,2080.198316,1.754741,2.912450e+07,1.116977e+08
min,2010.000000,3.860000,0.000000,0.000000,0.000000e+00,0.000000e+00
25%,2013.750000,7.840750,53.000000,5.600000,0.000000e+00,0.000000e+00
50%,2017.500000,10.913500,138.000000,6.300000,0.000000e+00,0.000000e+00
75%,2021.250000,17.336500,422.000000,6.923000,2.200000e+06,1.654473e+06
max,2025.000000,3876.006000,37119.000000,10.000000,4.600000e+08,2.799439e+09


# Valores faltantes

In [7]:
missing = pd.DataFrame({
    "Cantidad": df_movies.isnull().sum(),
    "Porcentaje (%)": (df_movies.isnull().mean() * 100).round(2)
})

missing.sort_values("Cantidad", ascending=False)

,Cantidad,Porcentaje (%)
duration,16000,100.00
country,466,2.91
cast,204,1.27
director,132,0.82
description,132,0.82
genres,107,0.67
type,0,0.00
show_id,0,0.00
title,0,0.00
date_added,0,0.00


# Revisar duplicados

In [8]:
print("Filas duplicadas:", df_movies.duplicated().sum())
print("Títulos duplicados:", df_movies["title"].duplicated().sum())

Filas duplicadas: 0
Títulos duplicados: 515


# Revisar valores 0 en presupuesto e ingresos

In [9]:
print("Budget = 0:", (df_movies["budget"] == 0).sum())
print("Revenue = 0:", (df_movies["revenue"] == 0).sum())

Budget = 0: 11153
Revenue = 0: 10355


# Tratamiento de genres

In [10]:
df_movies["genres"].isnull().sum()

np.int64(107)

# Crear dataset limpio

In [11]:
df_movies_clean = df_movies.copy()

print("Filas:", df_movies_clean.shape[0])
print("Columnas:", df_movies_clean.shape[1])

Filas: 16000
Columnas: 18


# Limpiar espacios en variables de texto

In [12]:
columnas_texto = df_movies_clean.select_dtypes(include="object").columns

for columna in columnas_texto:
    df_movies_clean[columna] = df_movies_clean[columna].str.strip()

# Revisar tipos de datos

In [13]:
df_movies_clean.dtypes

,0
show_id,int64
type,object
title,object
director,object
cast,object
country,object
date_added,object
release_year,int64
rating,float64
duration,float64


In [14]:
columnas_numericas = [
    "release_year",
    "popularity",
    "vote_count",
    "vote_average",
    "budget",
    "revenue"
]

for columna in columnas_numericas:
    df_movies_clean[columna] = pd.to_numeric(
        df_movies_clean[columna],
        errors="coerce"
    )

# Mantener generes faltantes

In [15]:
print("Géneros faltantes:", df_movies_clean["genres"].isna().sum())

Géneros faltantes: 107


# Revición del dataset

In [17]:
df_movies_clean.info()
df_movies_clean.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16000 entries, 0 to 15999
Data columns (total 18 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   show_id       16000 non-null  int64  
 1   type          16000 non-null  object 
 2   title         16000 non-null  object 
 3   director      15868 non-null  object 
 4   cast          15796 non-null  object 
 5   country       15534 non-null  object 
 6   date_added    16000 non-null  object 
 7   release_year  16000 non-null  int64  
 8   rating        16000 non-null  float64
 9   duration      0 non-null      float64
 10  genres        15893 non-null  object 
 11  language      16000 non-null  object 
 12  description   15868 non-null  object 
 13  popularity    16000 non-null  float64
 14  vote_count    16000 non-null  int64  
 15  vote_average  16000 non-null  float64
 16  budget        16000 non-null  int64  
 17  revenue       16000 non-null  int64  
dtypes: float64(4), int64(5), o

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,language,description,popularity,vote_count,vote_average,budget,revenue
0,10192,Movie,Shrek Forever After,Mike Mitchell,"Mike Myers, Eddie Murphy, Cameron Diaz, Antoni...",United States of America,2010-05-16,2010,6.380,NaN,"Comedy, Adventure, Fantasy, Animation, Family",en,A bored and domesticated Shrek pacts with deal...,203.893,7449,6.380,165000000,752600867
1,27205,Movie,Inception,Christopher Nolan,"Leonardo DiCaprio, Joseph Gordon-Levitt, Ken W...","United Kingdom, United States of America",2010-07-15,2010,8.369,NaN,"Action, Science Fiction, Adventure",en,"Cobb, a skilled thief who commits corporate es...",156.242,37119,8.369,160000000,839030630
2,12444,Movie,Harry Potter and the Deathly Hallows: Part 1,David Yates,"Daniel Radcliffe, Emma Watson, Rupert Grint, T...","United Kingdom, United States of America",2010-11-17,2010,7.744,NaN,"Adventure, Fantasy",en,"Harry, Ron and Hermione walk away from their l...",121.191,19327,7.744,250000000,954305868
3,38757,Movie,Tangled,"Byron Howard, Nathan Greno","Mandy Moore, Zachary Levi, Donna Murphy, Ron P...",United States of America,2010-11-24,2010,7.600,NaN,"Animation, Family, Adventure",en,"Feisty teenager Rapunzel, who has long and mag...",111.762,11638,7.600,260000000,592461732
4,10191,Movie,How to Train Your Dragon,"Chris Sanders, Dean DeBlois","Jay Baruchel, Gerard Butler, Craig Ferguson, A...",United States of America,2010-03-18,2010,7.800,NaN,"Fantasy, Adventure, Animation, Family",en,As the son of a Viking leader on the cusp of m...,110.044,13259,7.800,165000000,494879471


# Guardar dataset procesado

In [24]:
df_movies_clean.to_csv(
    "movies_clean.csv",
    index=False
)

print("Dataset procesado creado.")

from google.colab import files

files.download("movies_clean.csv")

Dataset procesado creado.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>